# Bias Detection using Longformer

This notebook implements a Longformer-based model for detecting political bias in news articles. The implementation includes:
- Loading data from the original Article-Bias-Prediction repository
- Training a Longformer model with memory-efficient batching
- Saving checkpoints to Google Drive
- Real-time training monitoring

## Setup
First, let's install the required packages and set up Google Drive access.

In [ ]:
# Install required packages
!pip install torch transformers numpy pandas scikit-learn matplotlib seaborn tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create directories for saving model checkpoints
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/bias_detection/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
# Clone the original repository
!git clone https://github.com/ramybaly/Article-Bias-Prediction.git
DATA_DIR = 'Article-Bias-Prediction/data/jsons'

## Imports and Constants

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import LongformerTokenizer, LongformerForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from IPython.display import display, clear_output

# Constants
RANDOM_STATE = 42
MAX_LENGTH = 4096  # Longformer supports up to 4096 tokens
BATCH_SIZE = 2     # Small batch size for memory efficiency
GRADIENT_ACCUMULATION_STEPS = 4  # Simulate larger batch size
EPOCHS = 10
LEARNING_RATE = 2e-5

# Ensure we're using GPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Set memory efficient options
torch.cuda.empty_cache()
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

## Helper Classes

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.best_model = None
        self.should_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = self._save_model_state(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.best_model = self._save_model_state(model)
            self.counter = 0
    
    def _save_model_state(self, model):
        return {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    def get_best_model(self, model):
        if self.best_model is not None:
            model.load_state_dict(self.best_model)
        return model

class ArticleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        global_attention_mask = torch.zeros_like(encoding['attention_mask'])
        global_attention_mask[:, 0] = 1
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'global_attention_mask': global_attention_mask.flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

## Data Loading and Processing

In [ ]:
def load_articles():
    articles = []
    for filename in os.listdir(DATA_DIR):
        if filename.endswith('.json'):
            with open(os.path.join(DATA_DIR, filename), 'r', encoding='utf-8') as f:
                article = json.load(f)
                articles.append(article)
    return pd.DataFrame(articles)

print("Loading articles...")
df = load_articles()
print(f"Loaded {len(df)} articles")

# Display class distribution
print("\nClass Distribution:")
print(df['bias'].value_counts())

## Training Functions

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(data_loader, desc='Training')
    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch['global_attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()
        total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        
        if (batch_idx + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        progress_bar.set_postfix({'loss': f'{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}'})
        
        del outputs
        torch.cuda.empty_cache()
    
    return total_loss / len(data_loader)

def evaluate_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            global_attention_mask = batch['global_attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            _, preds = torch.max(outputs.logits, dim=1)
            
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
            
            del outputs
            torch.cuda.empty_cache()
    
    return predictions, actual_labels, total_loss / len(data_loader)

## Model Training

In [ ]:
# Prepare data
texts = df['content'].tolist()
labels = df['bias'].tolist()

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels
)

# Initialize tokenizer and model
print("Initializing Longformer model and tokenizer...")
tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')

# Load model on CPU first
model = LongformerForSequenceClassification.from_pretrained(
    'allenai/longformer-base-4096',
    num_labels=3,
    attention_window=512,
    output_attentions=False,
    output_hidden_states=False
)

# Clear memory before moving to GPU
torch.cuda.empty_cache()
model.to(DEVICE)

# Create datasets and loaders
train_dataset = ArticleDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = ArticleDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# Initialize optimizer and early stopping
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
early_stopping = EarlyStopping(patience=3, min_delta=0.001)

# Training loop
print(f"Starting training on {DEVICE}...")
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    
    # Monitor GPU memory
    if DEVICE.type == 'cuda':
        print("\nGPU Memory Usage:")
        !nvidia-smi
    
    train_loss = train_epoch(model, train_loader, optimizer, DEVICE)
    print(f"Average training loss: {train_loss:.4f}")
    
    predictions, actual_labels, val_loss = evaluate_model(model, val_loader, DEVICE)
    
    # Early stopping check
    early_stopping(val_loss, model)
    
    # Calculate metrics
    accuracy = accuracy_score(actual_labels, predictions)
    print(f"\nValidation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(actual_labels, predictions))
    
    # Save checkpoint to Drive
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'accuracy': accuracy
    }
    torch.save(checkpoint, f"{CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.pt")
    
    if early_stopping.should_stop:
        print(f"\nEarly stopping triggered at epoch {epoch+1}")
        break

# Load the best model
model = early_stopping.get_best_model(model)

print("\nTraining completed!")

## Save Final Model

In [ ]:
# Save the final model to Drive
final_save_dir = f"{CHECKPOINT_DIR}/final_model"
os.makedirs(final_save_dir, exist_ok=True)

model.save_pretrained(final_save_dir)
tokenizer.save_pretrained(final_save_dir)

print(f"Final model saved to {final_save_dir}")